In [17]:
# Step 0 - Project Overview
# Key Concepts Used: 
# Agent powered by an LLM (simulated) – Classifies emergencies from text
# Sessions & Memory – Stores last 5 reports
# Sequential Multi-Agent System – Initial agent for classification, final agent for severity/advice

In [14]:
# Step 1: Import Libraries
from collections import deque
session_memory = deque(maxlen=5)
session_memory.clear() #to clear all previous or unwanted memory

In [15]:
#Step 2: Small Knowledge Base
emergency_types = {
    "injury": ["bleeding", "injured", "fracture", "wounded"],
    "stuck": ["stuck", "trapped", "caught"],
    "medical": ["unconscious", "dizzy", "weak"],
    "starvation": ["hungry", "thirsty", "dehydrated"],
    "threat": ["attacked", "aggressive", "bitten"]
}

severity_levels = {
    "injury": "High",
    "stuck": "Medium",
    "medical": "High",
    "starvation": "High",
    "threat": "Medium"
}

In [16]:
#Step 3: Classification
def classify(text):
    text = text.lower()
    for etype, keywords in emergency_types.items():
        if any(k in text for k in keywords):
            return etype
    return "unknown"

In [17]:
#Step 4: Advice & Follow-Up
def get_initial_advice(etype):
    if etype == "injury":
        return "Avoid moving the animal and call a vet or rescue team."
    if etype == "stuck":
        return "If safe, try to free it gently or call for help."
    if etype == "medical":
        return "Keep the animal calm and contact a vet immediately."
    if etype == "starvation":
        return "Offer small sips of water if safe and call animal care."
    if etype == "threat":
        return "Keep distance and call animal control."
    return "Observe from a safe distance."

def get_followup(etype):
    q = {
        "injury": "Is the animal conscious?",
        "stuck": "Can you reach the animal safely?",
        "medical": "Is the animal breathing normally?",
        "starvation": "Can the animal stand or move?",
        "threat": "Is the threat still nearby?"
    }
    return q.get(etype, "Any more details?")

In [18]:
#Step 5: Initial Agent 
def initial_agent(text):
    etype = classify(text)
    severity = severity_levels.get(etype, "Unknown")
    advice = get_initial_advice(etype)
    follow_q = get_followup(etype)

    session_memory.append({
        "text": text,
        "type": etype,
        "severity": severity,
        "finalized": False
    })

    return {
        "Emergency Type": etype.capitalize(),
        "Severity": severity,
        "Advice": advice,
        "Follow-up Question": follow_q
    }


In [19]:
#Step 6: Final Agent
def final_agent(user_reply):
    r = session_memory[-1]
    etype = r["type"]
    reply = user_reply.lower()

    if any(x in reply for x in ["unconscious", "not", "can't", "not breathing"]):
        new_severity = "Very High"
        meaning = "Immediate action recommended."
    else:
        new_severity = r["severity"]
        meaning = "Keep monitoring and seek help."

    final_advice = {
        "injury": "Do not move the animal. Call a vet or rescue urgently.",
        "stuck": "If safe, free the animal. Otherwise call rescue.",
        "medical": "Do not give food/water. Contact a vet immediately.",
        "starvation": "Offer only small water if alert. Call animal care.",
        "threat": "Stay away. Call animal control."
    }.get(etype, "Contact local rescue.")

    r["severity"] = new_severity
    r["finalized"] = True

    return f"""
Thank you for the update.
Updated Severity : {new_severity}
Meaning          : {meaning}
Final Advice: {final_advice}
"""


In [20]:
# Step 7: Demo Input (final clean interactive-style)
demo_inputs = [
    ("A dog is bleeding on the road and can't move.", "No, the dog is unconscious."),
    ("A pigeon is stuck behind my AC box.", "Yes, I can see it but cannot reach it safely."),
]

for i, (initial_text, user_followup) in enumerate(demo_inputs, 1):
    # Step 1: Show the input query first
    print(f"Input Query: {initial_text}\n")
    
    # Step 2: Initial Report (skip printing follow-up question here)
    print(f"Demo {i}:")
    init_out = initial_agent(initial_text)
    for k, v in init_out.items():
        if k != "Follow-up Question":
            print(f"{k}: {v}")
    
    # Step 3: Show follow-up question only once
    follow_up_q = init_out["Follow-up Question"]
    print(f"Follow-Up Question: {follow_up_q}")
    
    # Step 4: User reply
    user_reply = user_followup
    print(f"User Reply: {user_reply}\n")
    
    # Step 5: After Follow-Up
    print(f"Demo {i}: After Follow-Up")
    final_out = final_agent(user_reply)
    print(final_out)
    
    # Separator for clarity
    print("---"*24)

Input Query: A dog is bleeding on the road and can't move.

Demo 1:
Emergency Type: Injury
Severity: High
Advice: Avoid moving the animal and call a vet or rescue team.
Follow-Up Question: Is the animal conscious?
User Reply: No, the dog is unconscious.

Demo 1: After Follow-Up

Thank you for the update.
Updated Severity : Very High
Meaning          : Immediate action recommended.
Final Advice: Do not move the animal. Call a vet or rescue urgently.

------------------------------------------------------------------------
Input Query: A pigeon is stuck behind my AC box.

Demo 2:
Emergency Type: Stuck
Severity: Medium
Advice: If safe, try to free it gently or call for help.
Follow-Up Question: Can you reach the animal safely?
User Reply: Yes, I can see it but cannot reach it safely.

Demo 2: After Follow-Up

Thank you for the update.
Updated Severity : Very High
Meaning          : Immediate action recommended.
Final Advice: If safe, free the animal. Otherwise call rescue.

--------------

In [21]:
# Step 8: Print Session Memory
print("Session Memory (Last Reports):")
for idx, mem in enumerate(session_memory, 1):
    print(f"Report {idx}:")
    print(f"  Description   : {mem['text']}")
    print(f"  Emergency Type: {mem['type'].capitalize()}")
    print(f"  Severity      : {mem['severity']}")
    print(f"  Finalized     : {mem.get('finalized', False)}")
    print("-"*30)

Session Memory (Last Reports):
Report 1:
  Description   : A dog is bleeding on the road and can't move.
  Emergency Type: Injury
  Severity      : Very High
  Finalized     : True
------------------------------
Report 2:
  Description   : A pigeon is stuck behind my AC box.
  Emergency Type: Stuck
  Severity      : Very High
  Finalized     : True
------------------------------
